In [ ]:
## -1. Google Collab Set-up

In [ ]:
from google.colab import drive
import os

# 1. Mounting Google Drive: This allows Colab to access files in your Google Drive
drive.mount('/content/drive/')

# 2. Tell Colab where to find your assignment files and where to save your work

# TODO: Enter the relative path in your Google Drive of the assignment.
FOLDERNAME = "cs7643/CS7643-Final-Project-main" # e.g. 'cs7643/ps0/'

assert FOLDERNAME is not None, "[!] Enter the foldername."

FULL_PATH = "/content/drive/MyDrive/" + FOLDERNAME
assert os.path.exists(FULL_PATH), "Make sure your FOLDERNAME is correct"

In [ ]:
!pip install torch yfinance numpy matplotlib arch pandas

In [ ]:
#The setup
import yfinance as yf
from arch import arch_model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

stock_choice = "AAPL"

data = yf.download(
    stock_choice,
    start="2000-01-01",
    end="2021-12-31",
    interval="1d",
    auto_adjust=True
)

In [ ]:
#train regime

#GARCH
scaling_factor = 100 #the model gave a low number error
prices = data["Close"]
returns = np.log(prices).diff().dropna()

train_returns = returns["2000-01-01":"2017-12-31"]
val_returns  = returns["2018-01-01":"2018-12-31"]
print(len(val_returns))

model = arch_model(train_returns * scaling_factor, vol="Garch", p=1, q=1)
res = model.fit()
#print(res.summary())


In [ ]:
#validation regime
horizon = 20

#GARCH
garch_preds = []
targets = []

history = train_returns.copy() * scaling_factor

for t in range(len(val_returns) - horizon):

    #GARCH Refit
    model = arch_model(history, p=1, q=1)
    res = model.fit(disp="off")
    forecast = res.forecast(horizon=horizon)
    var_forecast = forecast.variance.values[-1]
    garch_vol = np.sqrt(np.mean(var_forecast)) / scaling_factor
    garch_preds.append(garch_vol)

    #Ground Truth
    future = val_returns.iloc[t:t+horizon]
    realized_vol = np.std(future.values)
    targets.append(realized_vol)

    #new history
    history = pd.concat([history, val_returns.iloc[t:t+1]*scaling_factor])


In [ ]:
#MSE error

garch_pred_np = np.array(garch_preds)
targets_np = np.array(targets)
GARCH_MSE = np.mean((garch_pred_np - targets_np)**2)
print("GARCH MSE:", GARCH_MSE)
#print(len(garch_pred_np))

plt.plot(garch_preds, label="GARCH Predicted Volatility")
plt.plot(targets, label="Realized Volatility (Ground Truth)")

plt.title("GARCH vs Ground Truth Volatility on AAPL (2018)")
plt.xlabel("Time Step")
plt.ylabel("Volatility")

plt.legend()
plt.grid(True)

In [ ]:
#GARCH but with 10 or 60 day lookbacks
#GARCH but with 10 or 60 day lookbacks
'''
I want a function that takes a range of prices, takes a year, and outputs a
graph of GARCH vs target volatility. The special thing is that the GARCH
should be contrainsted to certain lookbehind date (which is another passable param)

The volatility will be a 20 day future forecast and I'll want it for every day in
the year parameter. It will work for one ticker at a time
'''
def plot_garch_vs_target(price_series,
                         lookback,
                         year='2018',
                         horizon=20,
                         history_start=None,
                         scaling_factor=100.0,
                         fit_disp='off'):

    #convert prices
    returns = np.log(price_series).diff().dropna().sort_index()
    if history_start is not None:
        returns = returns.loc[history_start:]

    target_idx = returns.loc[f'{year}-01-01':f'{year}-12-31'].index
    dates = []
    preds = []
    targets = []

    for date in target_idx:
        pos = returns.index.get_loc(date)
        #check history and future
        if pos < lookback:
            continue
        if pos + horizon > len(returns):
            continue
        hist = returns.iloc[pos - lookback: pos].values.astype(float)
        fut  = returns.iloc[pos: pos + horizon].values.astype(float)
        target = float(np.log(np.std(fut)))

        pred = np.nan
        model = arch_model(hist * scaling_factor, vol='Garch', p=1, q=1)
        res = model.fit(disp=fit_disp)
        fc = res.forecast(horizon=horizon)
        var_forecast = fc.variance.values[-1]
        pred = float(np.log(np.sqrt(np.mean(var_forecast)) / scaling_factor))

        dates.append(pd.to_datetime(date))
        preds.append(pred)
        targets.append(target)

    dates = np.array(dates, dtype='datetime64[ms]')
    preds = np.array(preds, dtype=np.float32)
    targets = np.array(targets, dtype=np.float32)

    valid = ~np.isnan(preds)
    mse = np.mean((preds[valid] - targets[valid])**2) if valid.any() else np.nan
    print(f'lookback={lookback}  year={year}  horizon={horizon}  days={valid.sum()}  MSE={mse:.6e}')

    #time-series plot
    plt.figure(figsize=(12,4))
    if valid.any():
        plt.plot(dates[valid], preds[valid], label='GARCH pred', alpha=0.8)
        plt.plot(dates[valid], targets[valid], label='Realized (target)', alpha=0.9)
    else:
        plt.plot(dates, targets, label='Realized (target)', alpha=0.9)
    plt.title(f'GARCH vs Realized on AAPL: lookback={lookback} ({year})')
    plt.xlabel('Date'); plt.ylabel('Volatility'); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

    #return arrays
    return dates, preds, targets



In [ ]:
#Garch on 10/60 lookbacks tester

tickers = ['AAPL']
data = yf.download(
    tickers,
    start="2000-01-01",
    end="2021-12-31",
    interval="1d",
    auto_adjust=True
)

price_series = data["Close"]   # or prices variable you already have
dates, preds, targets = plot_garch_vs_target(price_series, lookback=10, year='2018', horizon=20, history_start='2017-01-01')

In [ ]:
dates60, preds60, targets60 = plot_garch_vs_target(price_series, lookback=60, year='2018', horizon=20, history_start='2017-01-01')


In [ ]:
## RNN Data from Checkpoints

In [ ]:
## -1. Google Collab Set-up

In [ ]:
from google.colab import drive
import os

# 1. Mounting Google Drive: This allows Colab to access files in your Google Drive
drive.mount('/content/drive/')

# 2. Tell Colab where to find your assignment files and where to save your work

# TODO: Enter the relative path in your Google Drive of the assignment.
FOLDERNAME = "cs7643/CS7643-Final-Project-main" # e.g. 'cs7643/ps0/'

assert FOLDERNAME is not None, "[!] Enter the foldername."

FULL_PATH = "/content/drive/MyDrive/" + FOLDERNAME
assert os.path.exists(FULL_PATH), "Make sure your FOLDERNAME is correct"

In [ ]:
%cd $FULL_PATH

In [ ]:
## Begin the grabbing of checkpoint files

In [ ]:
# 1) Setup & helper
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from test_data import prepare_data, TICKERS, VolatilityDataset
from torch.utils.data import DataLoader
from train import evaluate

# model classes
from RNNmodel import VanillaRNN
from LSTMRun.LSTMmodel import LSTMModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_state_from_path(path):
    ckpt = torch.load(path, map_location="cpu")
    # checkpoint likely is a state_dict (or wrapped dict)
    return ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))

def make_loader(returns, targets, lookback, batch_size=256):
    ds = VolatilityDataset(returns, targets, lookback=lookback)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    return loader

# 2) Data
returns, target_vol = prepare_data(force_download=False)   # uses CACHE_FILE if present
# We will examine year 2018 for AAPL (same as your GARCH)
ticker = "AAPL"
ticker_idx = TICKERS.index(ticker)

# 3) checkpoint paths (update if you uploaded elsewhere)
ckpt_candidates = {
    "rnn10":  os.path.join(FULL_PATH, "rnn_lookback10_best.pt"),
    "rnn60":  os.path.join(FULL_PATH, "rnn_lookback60_best.pt"),
    "lstm10": os.path.join(FULL_PATH, "LSTMRun", "lstm_lookback10_best.pt"),
    "lstm60": os.path.join(FULL_PATH, "LSTMRun", "lstm_lookback60_best.pt"),
}
for k,p in ckpt_candidates.items():
    if not os.path.exists(p):
        matches = glob.glob(os.path.join(FULL_PATH, "**", os.path.basename(p)), recursive=True)
        ckpt_candidates[k] = matches[0] if matches else None

print("Checkpoint files:", ckpt_candidates)

# 4) Evaluate function that returns per-sample preds + date + stock index
def eval_for_model(kind, lookback, ckpt_path):
    if ckpt_path is None:
        return None
    # prepare loader for this lookback
    loader = make_loader(returns, target_vol, lookback, batch_size=256)
    # instantiate model matching training hyperparams
    if kind == "rnn":
        model = VanillaRNN(num_stocks=len(TICKERS), embed_dim=16, hidden_size=16, num_layers=1, dropout=0.1)
    else:  # "lstm"
        model = LSTMModel(num_stocks=len(TICKERS), embed_dim=16, hidden_size=64, num_layers=2, dropout=0.1)
    sd = load_state_from_path(ckpt_path)
    model.load_state_dict(sd)
    model.to(device)
    metrics = evaluate(model, loader, device)   # uses loader order (shuffle=False)
    preds_log = metrics["preds_log"]
    targets_log = metrics["targets_log"]

    # build arrays of dates & stock indices from dataset.samples (same order)
    samples = loader.dataset.samples  # list of (t, s)
    dates = np.array([returns.index[t] for (t, s) in samples], dtype="datetime64[ms]")
    stocks = np.array([s for (t, s) in samples], dtype=int)

    return dict(preds_log=preds_log, targets_log=targets_log, dates=dates, stocks=stocks, metrics=metrics)

# 5) Run eval for each available checkpoint
results = {}
specs = [("rnn",10,"rnn10"), ("rnn",60,"rnn60"), ("lstm",10,"lstm10"), ("lstm",60,"lstm60")]
for kind,look,ckk in specs:
    path = ckpt_candidates.get(ckk)
    print(f"Evaluating {kind}{look} -> {path}")
    results[f"{kind}{look}"] = eval_for_model(kind, look, path)

# 6) Extract AAPL 2018 series for each model (convert to original vol)
start, end = np.datetime64("2018-01-01"), np.datetime64("2018-12-31")
aapl_series = {}
for key, res in results.items():
    if res is None:
        continue
    mask = (res["stocks"] == ticker_idx) & (res["dates"] >= start) & (res["dates"] <= end)
    if not mask.any():
        continue
    # preds/targets order matches mask
    preds_vol = np.exp(res["preds_log"][mask])
    targets_vol = np.exp(res["targets_log"][mask])
    dates = res["dates"][mask]
    # sort by date
    order = np.argsort(dates)
    aapl_series[key] = (dates[order], preds_vol[order], targets_vol[order])

# 7) Plot: overlay model predictions + realized vol + (optional) your GARCH arrays
plt.figure(figsize=(14,5))
# plot realized vol from model targets (they should match the RNN/LSTM target calc)
if aapl_series:
    # pick one model to plot realized (they all use same target), use rnn10 if exists, else first
    pick = next(iter(aapl_series))
    dates, _, realized = aapl_series[pick]
    plt.plot(dates, realized, label="Realized (target)", color="black", linewidth=1.2)

# plot each model's preds
colors = {"rnn10":"C0","rnn60":"C1","lstm10":"C2","lstm60":"C3"}
for key,(dates,preds,_) in aapl_series.items():
    plt.plot(dates, preds, label=f"{key} pred", color=colors.get(key,"C7"), alpha=0.9)

# If you ran your GARCH function earlier and have `dates` & `preds`/`targets` variables from it:
# - for lookback 10 you earlier saved as (dates, preds, targets)
# - for lookback 60 perhaps as (dates60, preds60, targets60)
try:
    if 'dates' in globals() and 'preds' in globals():
        # convert from log-vol -> vol
        garch_dates = np.array(dates, dtype="datetime64[ms]")
        plt.plot(garch_dates, np.exp(preds), label="GARCH (lookback10)", color="C5", linestyle="--")
    if 'dates60' in globals() and 'preds60' in globals():
        garch_dates60 = np.array(dates60, dtype="datetime64[ms]")
        plt.plot(garch_dates60, np.exp(preds60), label="GARCH (lookback60)", color="C6", linestyle="--")
except Exception as e:
    print("No GARCH series available in globals()", e)

plt.xlabel("Date"); plt.ylabel("Volatility"); plt.title(f"{ticker} 2018: Model preds vs Realized & GARCH")
plt.legend(ncol=2); plt.grid(True); plt.tight_layout(); plt.show()